In [1]:
# Celda 1
%cd C:/Users/fabi_/Documents/workflow/textos-catolicos-xix

import os
import pickle
import re
import sys
import tempfile
from collections import Counter
from multiprocessing import Pool

sys.path.insert(0, "ocr-piloto")
import reglas_modernizacion_postcorreccion as reglas  # Rules and their sources: see the top of that file

CLEAN_FOLDER = "textos-limpios"           # From 5-limpieza
MODERN_FOLDER = "textos-modernizados"     # Same files, modernized and post-corrected
REGISTER_FILE = "registro-modernizacion.csv"  # Every changed word: book, line, before, after, rule
REGISTER_FOLDER = os.path.join(tempfile.gettempdir(), "registro-modernizacion")  # One file per book and step, joined in cell 5
WORKERS = 3                               # Processes at once; each one holds the corpus vocabulary in memory

C:\Users\fabi_\Documents\workflow\textos-catolicos-xix


In [2]:
# Celda 2
# Vocabulary of the whole corpus (words, occurrences and books), from the clean texts. About a minute:
vocabulary = reglas.contar_vocabulario(CLEAN_FOLDER)
# Frequent OCR errors ("eon" -> "con"), left for step 5; words of the Spanish dictionary are never among them:
vocabulary["frecuentes"] = reglas.errores_frecuentes(vocabulary, "ocr-piloto/diccionario-es/es_ES")
# Drop caps misread at the beginning of a line ("Eué" -> "Fué"), with the pairs of words needed to choose between
# corrections and the names found inside lines, which are left alone. About 2 minutes:
vocabulary["capitulares"] = reglas.capitulares(vocabulary, "ocr-piloto/diccionario-es/es_ES")
needed_pairs, vocabulary["nombres_capitulares"] = reglas.pares_capitulares(CLEAN_FOLDER, vocabulary)
vocabulary["pares_capitulares"] = reglas.contar_pares(CLEAN_FOLDER, needed_pairs)
# Latin words outside the passages marked by 5-limpieza (short quotes): left as printed, like the marked passages:
vocabulary["latin"] = reglas.lexico_latino(CLEAN_FOLDER, vocabulary, "ocr-piloto/diccionario-es/es_ES")
VOCABULARY_FILE = os.path.join(tempfile.gettempdir(), "vocabulario-corpus.pkl")  # For the worker processes
with open(VOCABULARY_FILE, "wb") as f:
    pickle.dump(vocabulary, f)
print(len(vocabulary["freq"]), "different words,", len(vocabulary["abrev"]), "abbreviations,",
      len(vocabulary["frecuentes"]), "frequent OCR errors,",
      len(vocabulary["capitulares"]), "possible drop caps,",
      len(vocabulary["latin"]), "Latin words")

321099 different words, 302 abbreviations, 2493 frequent OCR errors, 10348 possible drop caps, 2030 Latin words


In [3]:
# Celda 3
# Modernization (accents, old spellings) and post-correction steps 1-4 and 6 (contractions, "sc", missing accents,
# glued words, space before punctuation), book by book, after moving punctuation at the beginning of a line to the end
# of the previous one and fixing drop caps. Each book also writes its changes to REGISTER_FOLDER. A book already done is skipped, so the cell can be stopped
# and run again; to redo everything, empty the output folder first. About 90 minutes with 2 processes:
if __name__ == "__main__":
    os.makedirs(MODERN_FOLDER, exist_ok=True)
    os.makedirs(REGISTER_FOLDER, exist_ok=True)
    jobs = []
    for file_name in sorted(os.listdir(CLEAN_FOLDER)):
        if file_name.endswith(".txt") and not os.path.exists(os.path.join(MODERN_FOLDER, file_name)):
            jobs.append((os.path.join(CLEAN_FOLDER, file_name), os.path.join(MODERN_FOLDER, file_name),
                         os.path.join(REGISTER_FOLDER, file_name[:-4] + ".1.csv")))
    totals = Counter()
    with Pool(WORKERS, initializer=reglas.iniciar, initargs=(VOCABULARY_FILE,)) as pool:
        for file_name, stats in pool.imap_unordered(reglas.modernizar_libro, jobs):
            totals.update(stats)
            print(file_name, stats)
    print("Total:", dict(totals))

agreda-mistica-ciudad-de-dios-tomo1.txt {'tilde': 6, 'contracción': 1, 'sc': 2}


agreda-cartas-a-felipe-iv-tomo1.txt {'tilde': 61, 'contracción': 174}


agreda-mistica-ciudad-de-dios-tomo2.txt {'tilde': 43, 'contracción': 12}


agreda-cartas-a-felipe-iv-tomo2.txt {'tilde': 120, 'contracción': 203}


agreda-mistica-ciudad-de-dios-tomo3.txt {'tilde': 52, 'contracción': 20, 'sc': 1}


agreda-mistica-ciudad-de-dios-tomo4.txt {'tilde': 72, 'contracción': 41, 'sc': 1}


... (135 outputs left out)

varios-romancero-y-cancionero-sagrados.txt {'sc': 76, 'tilde': 259, 'contracción': 330}


venegas-agonia-del-transito-de-la-muerte.txt {'contracción': 464, 'sc': 527, 'tilde': 43}


yepes-vida-de-santa-teresa-de-jesus.txt {'tilde': 97, 'contracción': 24}


zarate-discursos-de-la-paciencia-cristiana.txt {'contracción': 1290, 'tilde': 108, 'sc': 14}
Total: {'tilde': 7899, 'contracción': 21144, 'sc': 8968}


In [4]:
# Celda 4
# Post-correction step 5: OCR errors one typical confusion away from a common word ("ejernplo" -> "ejemplo"),
# only if the corrected word appears in the corpus next to the same neighbor, or if it is the only correction of a
# frequent OCR error ("mueho" -> "mucho"). First, the pairs of neighboring words that are needed; then, one pass over
# the corpus to count them; then, the corrections. Run it once after cell 3: a second run finds nothing left to
# correct and would leave the register of this step empty:
needed = set()
for file_name in sorted(os.listdir(MODERN_FOLDER)):
    with open(os.path.join(MODERN_FOLDER, file_name), encoding="utf-8") as f:
        for line in f:
            reglas.pares_necesarios(line.rstrip("\n"), vocabulary, needed)
pairs = reglas.contar_pares(CLEAN_FOLDER, needed)

PROTECTED = reglas.patron_protegido(vocabulary)  # Tags, abbreviations, tokens with digits: left as printed
stats = Counter()
for file_name in sorted(os.listdir(MODERN_FOLDER)):
    reglas.corregir_libro(os.path.join(MODERN_FOLDER, file_name), vocabulary, pairs, PROTECTED, stats,
                          os.path.join(REGISTER_FOLDER, file_name[:-4] + ".2.csv"))
print(stats["corregida"], "OCR errors corrected,", stats["frecuente"], "of them frequent errors without neighbors")


50325 OCR errors corrected, 7273 of them frequent errors without neighbors


In [5]:
# Celda 5
# The register of changes of the whole corpus, one row per changed word, in the order of the books and lines:
# book, line, before, after, rule. Rules: "modernización", "contracción", "sc", "tilde", "capitular",
# "puntuación al inicio de línea", "palabras pegadas", "OCR (vecino)", "OCR (sin vecino)", "OCR (frecuente)".
# Filter by rule to look for false corrections. Step 6 (space before punctuation) is not registered:
import csv
rows = []
for file_name in os.listdir(REGISTER_FOLDER):
    with open(os.path.join(REGISTER_FOLDER, file_name), encoding="utf-8", newline="") as f:
        rows.extend(csv.reader(f))
rows.sort(key=lambda row: (row[0], int(row[1])))
with open(REGISTER_FILE, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["libro", "linea", "antes", "despues", "regla"])
    writer.writerows(rows)
print(len(rows), "changes in", REGISTER_FILE)
print(Counter(row[4].split(" (pasa")[0] for row in rows).most_common())


1019552 changes in registro-modernizacion.csv


[('modernización', 910709), ('OCR (vecino)', 36362), ('contracción', 21144), ('palabras pegadas', 10490), ('puntuación al inicio de línea', 9536), ('sc', 8960), ('tilde', 7891), ('OCR (frecuente)', 7273), ('OCR (sin vecino)', 6690), ('capitular', 489), ('sc+tilde', 8)]
